In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import warnings


warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

print("--- [START] Preprocessing Script ---")


file_path = '/kaggle/input/lending-club/accepted_2007_to_2018Q4.csv.gz'
df_sample = pd.read_csv(file_path, nrows=200000)
print(f"Loaded {len(df_sample)} rows.")


bad_loan_statuses = ['Charged Off', 'Late (31-120 days)', 'Late (16-30 days)', 'Default']
good_loan_status = 'Fully Paid'

df_sample['is_default'] = np.where(
    df_sample['loan_status'].isin(bad_loan_statuses), 1,
    np.where(df_sample['loan_status'] == good_loan_status, 0, -1) # -1 for statuses we'll drop
)


df = df_sample[df_sample['is_default'].isin([0, 1])].copy()
df.reset_index(drop=True, inplace=True)

print(f"Filtered to {len(df)} rows with known outcomes.")
print(df['is_default'].value_counts())


min_nulls = len(df) * 0.4
cols_to_drop_nulls = df.columns[df.isnull().sum() > min_nulls]
df = df.drop(columns=cols_to_drop_nulls)

cols_to_drop_manual = [
    'id', 'url', 'title', 'emp_title', 'zip_code', 'addr_state',
    'loan_status'
]
df = df.drop(columns=cols_to_drop_manual, errors='ignore')

print(f"Shape after dropping null/irrelevant: {df.shape}")

leaky_cols = [
    'funded_amnt', 'funded_amnt_inv', 'installment',
    'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low',
    'debt_settlement_flag'
]
df = df.drop(columns=leaky_cols, errors='ignore')

print(f"Shape after dropping leaky columns: {df.shape}")


df['term'] = df['term'].str.replace(' months', '').str.strip().astype(int)


df['emp_length'] = df['emp_length'].str.replace('< 1 year', '0 years')
df['emp_length'] = df['emp_length'].str.replace('10+ years', '10 years')
df['emp_length'] = df['emp_length'].str.replace(' years', '').str.replace(' year', '')
df['emp_length'] = df['emp_length'].fillna('0')
df['emp_length'] = df['emp_length'].astype(int)


df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], errors='coerce')
reference_date = pd.to_datetime('2019-01-01')
df['credit_history_months'] = ((reference_date - df['earliest_cr_line']).dt.days / 30).astype(float)
df = df.drop(columns=['earliest_cr_line'], errors='ignore')

print("Engineered 'term', 'emp_length', 'credit_history_months'.")


numeric_features = [
    'loan_amnt', 'int_rate', 'term', 'emp_length', 'annual_inc',
    'dti', 'fico_range_low', 'open_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'total_acc', 'credit_history_months'
]
categorical_features = [
    'grade', 'home_ownership', 'verification_status', 'purpose'
]
target_col = 'is_default' 

features = numeric_features + categorical_features
df_final = df[features + [target_col]].copy()

for col in numeric_features:
    if df_final[col].isnull().sum() > 0:
        median_val = df_final[col].median()
        df_final[col] = df_final[col].fillna(median_val)
        
print(f"Final data shape before encoding: {df_final.shape}")
print(f"Remaining nulls: {df_final.isnull().sum().sum()}")


df_processed = pd.get_dummies(df_final, columns=categorical_features, drop_first=True)
print(f"Shape after one-hot encoding: {df_processed.shape}")


X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]


reward_features_unscaled = df_final.loc[X.index, ['loan_amnt', 'int_rate']]


X_train, X_test, y_train, y_test, rewards_train_unscaled, rewards_test_unscaled = train_test_split(
    X, y, reward_features_unscaled, test_size=0.2, random_state=42, stratify=y
)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)


output_dir = '/kaggle/working/'


X_train_scaled.to_csv(output_dir + 'X_train_scaled.csv', index=False)
X_test_scaled.to_csv(output_dir + 'X_test_scaled.csv', index=False)
y_train.to_csv(output_dir + 'y_train.csv', index=False)
y_test.to_csv(output_dir + 'y_test.csv', index=False)


rewards_train_unscaled.to_csv(output_dir + 'rewards_train_unscaled.csv', index=False)
rewards_test_unscaled.to_csv(output_dir + 'rewards_test_unscaled.csv', index=False)

joblib.dump(scaler, output_dir + 'scaler.joblib')

print("\n--- [SUCCESS] ---")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")
print("All files saved to /kaggle/working/")

--- [START] Preprocessing Script ---
Loaded 200000 rows.
Filtered to 177016 rows with known outcomes.
is_default
0    140992
1     36024
Name: count, dtype: int64
Shape after dropping null/irrelevant: (177016, 87)
Shape after dropping leaky columns: (177016, 69)
Engineered 'term', 'emp_length', 'credit_history_months'.
Final data shape before encoding: (177016, 18)
Remaining nulls: 0
Shape after one-hot encoding: (177016, 37)

--- [SUCCESS] ---
X_train_scaled shape: (141612, 36)
X_test_scaled shape:  (35404, 36)
All files saved to /kaggle/working/


In [3]:
df_sample.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term,is_default
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,-1
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN,0


In [4]:
df.head()

,loan_amnt,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,...,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,disbursement_method,is_default,credit_history_months
0,3600.0,36,13.99,C,C4,10,MORTGAGE,55000.0,Not Verified,Dec-2015,...,0.0,0.0,178050.0,7746.0,2400.0,13734.0,N,Cash,0,187.733333
1,24700.0,36,11.99,C,C1,10,MORTGAGE,65000.0,Not Verified,Dec-2015,...,0.0,0.0,314017.0,39475.0,79300.0,24667.0,N,Cash,0,232.366667
2,20000.0,60,10.78,B,B4,10,MORTGAGE,63000.0,Not Verified,Dec-2015,...,0.0,0.0,218418.0,18696.0,6200.0,14877.0,N,Cash,0,224.233333
3,10400.0,60,22.45,F,F1,3,MORTGAGE,104433.0,Source Verified,Dec-2015,...,0.0,0.0,439570.0,95768.0,20300.0,88097.0,N,Cash,0,250.633333
4,11950.0,36,13.44,C,C3,4,RENT,34000.0,Source Verified,Dec-2015,...,0.0,0.0,16900.0,12798.0,9400.0,4000.0,N,Cash,0,380.500000


In [5]:
df_final.head()

,loan_amnt,int_rate,term,emp_length,annual_inc,dti,fico_range_low,open_acc,pub_rec,revol_bal,revol_util,total_acc,credit_history_months,grade,home_ownership,verification_status,purpose,is_default
0,3600.0,13.99,36,10,55000.0,5.91,675.0,7.0,0.0,2765.0,29.7,13.0,187.733333,C,MORTGAGE,Not Verified,debt_consolidation,0
1,24700.0,11.99,36,10,65000.0,16.06,715.0,22.0,0.0,21470.0,19.2,38.0,232.366667,C,MORTGAGE,Not Verified,small_business,0
2,20000.0,10.78,60,10,63000.0,10.78,695.0,6.0,0.0,7869.0,56.2,18.0,224.233333,B,MORTGAGE,Not Verified,home_improvement,0
3,10400.0,22.45,60,3,104433.0,25.37,695.0,12.0,0.0,21929.0,64.5,35.0,250.633333,F,MORTGAGE,Source Verified,major_purchase,0
4,11950.0,13.44,36,4,34000.0,10.20,690.0,5.0,0.0,8822.0,68.4,6.0,380.500000,C,RENT,Source Verified,debt_consolidation,0
